In [1]:
!pip install qiskit-ionq

zsh:1: command not found: pip


In [2]:
# Readiness check: load .env, verify API key, list IonQ backends
from dotenv import load_dotenv
import os
from qiskit_ionq import IonQProvider

# Load environment variables from .env if present
load_dotenv()

key = os.getenv("IONQ_API_KEY")
print("IONQ_API_KEY set:", bool(key))

if not key:
    print("Missing IONQ_API_KEY. Set it in your environment or .env and restart the kernel.")
else:
    try:
        prov = IonQProvider(key)
        print("Available backends:")
        for b in prov.backends():
            print("-", b.name)
    except Exception as e:
        print("Backend listing failed:", e)


/Users/jasonli/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


IONQ_API_KEY set: True


Available backends:
- <bound method BackendV1.name of <IonQSimulatorBackend('ionq_simulator')>>
- <bound method BackendV1.name of <IonQQPUBackend('ionq_qpu')>>


In [3]:
# Updated imports
import random
import hashlib
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_ionq import IonQProvider
from typing import List, Tuple
from qiskit_aer import AerSimulator
import numpy as np
import os

# Initialize IonQ provider
# Set IONQ_API_KEY environment variable or pass directly
provider = IonQProvider(os.getenv("IONQ_API_KEY"))


# UNCOMMENT BELOW IF YOU WANT TO CHECK WHICH BACKENDS ARE AVAILABLE RIGHT NOW
# def test_backend_connection(backend_name): # Checks which backends are available right now
#     """Test if a specific backend works"""
#     try:
#         backend = provider.get_backend(backend_name)
#         print(f"✅ Backend '{backend_name}' exists")
#         
#         # Test with minimal circuit
#         from qiskit import QuantumCircuit
#         qc = QuantumCircuit(1, 1)
#         qc.measure(0, 0)
#         
#         print(f"   Testing with minimal circuit...")
#         job = backend.run(qc, shots=1)
#         result = job.result()
#         print(f"   ✅ '{backend_name}' works successfully!")
#         return True
#         
#     except Exception as e:
#         print(f"   ❌ '{backend_name}' failed: {e}")
#         return False
# # Test all available backends
# print("Testing all available backends:")
# available_backends = provider.backends()
# for backend in available_backends:
#     test_backend_connection(backend.name)

# UNCOMMENT BELOW BASED ON WHCIH BACKEND YOU ARE WORKING WITH

backend = provider.get_backend("qpu.aria-1")  # IonQ real hardware
flag = 1 # flag to tell when it is in simulator vs. actual quantum hardware, 1 means quantum hardware

# backend = provider.get_backend("simulator")
# flag = 0 # simulator

# backend = AerSimulator()
# flag = 0 # flag to tell when it is in simulator vs. actual quantum hardware, 0 means AerSimulator

In [4]:
def random_bits(n: int) -> List[int]:
    return [random.randint(0, 1) for _ in range(n)]

def random_bases(n: int) -> List[int]:
    return [random.randint(0, 1) for _ in range(n)]

def prepare_batch_circuit(bits: List[int], bases: List[int]) -> QuantumCircuit:
    """Create a single circuit with multiple qubits for batch processing"""
    n = len(bits)
    qc = QuantumCircuit(n, n)
    
    for i, (bit, basis) in enumerate(zip(bits, bases)):
        if bit == 1:
            qc.x(i)
        if basis == 1:
            qc.h(i)
    
    return qc

def add_noise_to_circuit(qc: QuantumCircuit, noise_level: float = 0.05, noise_type: str = "depolarizing") -> QuantumCircuit:
    """Applies simulated noise to each qubit in the circuit."""
    noisy_qc = qc.copy()

    for i in range(noisy_qc.num_qubits):
        if random.random() < noise_level:
            if noise_type == "bitflip":
                noisy_qc.x(i)
            elif noise_type == "phaseflip":
                noisy_qc.z(i)
            elif noise_type == "depolarizing":
                gate = random.choice(['x', 'y', 'z'])
                getattr(noisy_qc, gate)(i)

    return noisy_qc


def measure_batch_circuit(base_qc: QuantumCircuit, bases: List[int]) -> QuantumCircuit:
    """Add measurement operations for Bob's bases"""
    n = base_qc.num_qubits
    qc = base_qc.copy()
    
    for i, basis in enumerate(bases):
        if basis == 1:
            qc.h(i)
        qc.measure(i, i)
    
    return qc

def run_circuit_get_bits(qc: QuantumCircuit, shots: int = 100) -> List[int]:
    """Run circuit on IonQ and return measured bits"""
    t_qc = transpile(qc, backend=backend, optimization_level=1)
    job = backend.run(t_qc, shots=shots)
    result = job.result()
    counts = result.get_counts()
    if shots == 1:
        outcome = list(counts.keys())[0]
        return [int(bit) for bit in outcome[::-1]]
    most_frequent = max(counts, key=counts.get)
    return [int(bit) for bit in most_frequent[::-1]]

def submit_circuit(qc: QuantumCircuit, shots: int = 100) -> str:
    t_qc = transpile(qc, backend=backend, optimization_level=1)
    job = backend.run(t_qc, shots=shots)
    print(f"IonQ job submitted. Job ID: {job.job_id()}")
    return job.job_id()

def eve_intercept_resend_batch(alice_bits: List[int], alice_bases: List[int], eve_bases: List[int]) -> QuantumCircuit:
    """Eve intercepts and resends using batch processing"""
    alice_circuit = prepare_batch_circuit(alice_bits, alice_bases)
    eve_measure_circuit = measure_batch_circuit(alice_circuit, eve_bases)
    eve_measurements = run_circuit_get_bits(eve_measure_circuit)
    return prepare_batch_circuit(eve_measurements, eve_bases)

def sift_key(alice_bits: List[int], alice_bases: List[int], bob_bits: List[int], bob_bases: List[int]) -> Tuple[List[int], List[int], List[int]]:
    """Return (sifted_alice, sifted_bob, indices_kept) where we keep positions with matching bases."""
    sifted_a = []
    sifted_b = []
    indices = []
    for i, (abits, abases, bbits, bbases) in enumerate(zip(alice_bits, alice_bases, bob_bits, bob_bases)):
        if alice_bases[i] == bob_bases[i]:
            sifted_a.append(alice_bits[i])
            sifted_b.append(bob_bits[i])
            indices.append(i)
    return sifted_a, sifted_b, indices

def error_rate(a_bits: List[int], b_bits: List[int]) -> float:
    if not a_bits:
        return 0.0
    mismatches = sum(x != y for x, y in zip(a_bits, b_bits))
    return mismatches / len(a_bits)

def cascade_error_correction(alice_key: List[int], bob_key: List[int], num_passes: int = 4) -> Tuple[List[int], List[int], int]:
    """Simplified Cascade protocol for BB84 error correction."""
    corrected_bob = bob_key.copy()
    total_corrections = 0

    key_length = len(alice_key)
    if key_length == 0:
        return alice_key, bob_key, 0

    for p in range(num_passes):
        block_size = max(1, key_length // (2 ** (p + 1)))
        indices = list(range(0, key_length, block_size))
        random.shuffle(indices)

        for start in indices:
            end = min(start + block_size, key_length)
            a_block = alice_key[start:end]
            b_block = corrected_bob[start:end]

            if sum(a_block) % 2 != sum(b_block) % 2:
                left, right = 0, len(a_block) - 1
                while left < right:
                    mid = (left + right) // 2
                    if (sum(a_block[:mid+1]) % 2) != (sum(b_block[:mid+1]) % 2):
                        right = mid
                    else:
                        left = mid + 1
                corrected_bob[start + left] ^= 1
                total_corrections += 1

    return alice_key, corrected_bob, total_corrections

def privacy_amplification(shared_key: List[int], final_length: int = None) -> List[int]:
    bitstring = ''.join(map(str, shared_key))
    hashed = hashlib.sha256(bitstring.encode()).hexdigest()
    hashed_bits = bin(int(hashed, 16))[2:].zfill(256)
    if final_length is None:
        final_length = len(shared_key) // 2
    final_bits = [int(b) for b in hashed_bits[:final_length]]
    return final_bits


def create_full_bb84_circuit(alice_bits: List[int], alice_bases: List[int], bob_bases: List[int]) -> QuantumCircuit:
    n = len(alice_bits)
    qc = QuantumCircuit(n, n)
    for i in range(n):
        if alice_bases[i] == 0:
            if alice_bits[i] == 1:
                qc.x(i)
        else:
            if alice_bits[i] == 0:
                qc.h(i)
            else:
                qc.x(i)
                qc.h(i)
    qc.barrier()
    for i in range(n):
        if bob_bases[i] == 1:
            qc.h(i)
    qc.measure(range(n), range(n))
    return qc

def run_bb84(n: int = 8, with_eve: bool = False, verbose: bool = True, print_circuits: bool = False, 
             noise_level: float = 0.0, noise_type: str = "depolarizing", cascade: bool = False, privacy_amp: bool = False):
    if n > 11 and flag:
        print(f"Warning: Reducing n from {n} to 8 due to hardware limitations")
        n = 8
    
    alice_bits = random_bits(n)
    alice_bases = random_bases(n)

    if with_eve:
        eve_bases = random_bases(n)
        print("\n" + "="*50)
        print("EVE INTERCEPT-RESEND ATTACK:")
        print("="*50)
        print(f"Eve's bases: {eve_bases}")
        forwarded_circuit = eve_intercept_resend_batch(alice_bits, alice_bases, eve_bases)
    else:
        forwarded_circuit = prepare_batch_circuit(alice_bits, alice_bases)

    if noise_level > 0:
        print("\n" + "="*50)
        print("ADDING SIMULATED NOISE TO CHANNEL")
        print("="*50)
        print(f"Noise level: {noise_level:.2f}, Type: {noise_type}")
        forwarded_circuit = add_noise_to_circuit(forwarded_circuit, noise_level, noise_type)

    bob_bases = random_bases(n)
    bob_measure_circuit = measure_batch_circuit(forwarded_circuit, bob_bases)

    if print_circuits:
        full_circuit = create_full_bb84_circuit(alice_bits, alice_bases, bob_bases)
        print("\n" + "="*50)
        print("COMPLETE BB84 CIRCUIT DIAGRAM:")
        print("="*50)
        print(full_circuit)
        print("\n")
        print(f"Total qubits: {full_circuit.num_qubits}")
        print(f"Circuit depth: {full_circuit.depth()}")
        print(f"Gate counts: {dict(full_circuit.count_ops())}")
        print("\n")

    if flag:
        job_id = submit_circuit(bob_measure_circuit)
        if verbose:
            print(f"Using backend: {backend.name}")
            print(f"Submitted IonQ job ID: {job_id}")
        # Return full context so downstream can finish the protocol after results arrive
        return {
            "job_id": job_id,
            "backend": backend.name,
            "alice_bits": alice_bits,
            "alice_bases": alice_bases,
            "bob_bases": bob_bases,
            "n": n,
            "with_eve": with_eve,
            "cascade": cascade,
            "privacy_amp": privacy_amp,
        }

    bob_bits = run_circuit_get_bits(bob_measure_circuit)

    sifted_a, sifted_b, indices = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    err = error_rate(sifted_a, sifted_b)

    if cascade:
        sifted_a_corr, sifted_b_corr, corrections = cascade_error_correction(sifted_a, sifted_b, num_passes = 4)
        err_after = error_rate(sifted_a_corr, sifted_b_corr)

    if privacy_amp:
        final_key = privacy_amplification(sifted_a_corr)

    if verbose:
        print(f"Using backend: {backend.name}")
        print(f"Alice bits:      {alice_bits}")
        print(f"Alice bases:     {alice_bases}")
        print(f"Bob bits:        {bob_bits}")
        print(f"Bob bases:       {bob_bases}")
        print(f"Sifted indices:  {indices}")
        print(f"Sifted Alice:    {sifted_a}")
        print(f"Sifted Bob:      {sifted_b}")
        if cascade:
            print("\n")
            print("With Cascade:")
            print(f"Error rate before Cascade: {err}")
            print(f"Total bit corrections: {corrections}")
            print(f"Error rate after Cascade:  {err_after}")
        else:
            print(f"Error rate:      {err}")
        print(f"Key length:      {len(sifted_a)}")
        if privacy_amp:
            print("\n")
            print("With Privacy Amplification:")
            print(f"Final key after Privacy Amplification: {final_key}")
            print(f"Final key length after Privacy Amplification: {len(final_key)}")
        

    return {
        'alice_bits': alice_bits,
        'alice_bases': alice_bases,
        'bob_bits': bob_bits,
        'bob_bases': bob_bases,
        'sifted_alice': sifted_a,
        'sifted_bob': sifted_b,
        'sifted_indices': indices,
        'error_rate': err,
        'error_rate_before': err,
        'corrections': corrections
    }

# Test
print("Testing BB84:")
result = run_bb84(n = 4, with_eve = False, verbose = True, print_circuits = True, noise_level = 0.0, noise_type = "depolarizing", cascade = True, privacy_amp = True)

Testing BB84:

COMPLETE BB84 CIRCUIT DIAGRAM:
           ░      ┌─┐      
q_0: ──────░──────┤M├──────
     ┌───┐ ░ ┌───┐└╥┘┌─┐   
q_1: ┤ X ├─░─┤ H ├─╫─┤M├───
     ├───┤ ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ X ├─░─┤ H ├─╫──╫─┤M├
     └───┘ ░ └┬─┬┘ ║  ║ └╥┘
q_3: ──────░──┤M├──╫──╫──╫─
           ░  └╥┘  ║  ║  ║ 
c: 4/══════════╩═══╩══╩══╩═
               3   0  1  2 


Total qubits: 4
Circuit depth: 3
Gate counts: {'measure': 4, 'x': 2, 'h': 2, 'barrier': 1}




/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for mcx can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for cnot can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for toffoli can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153:

IonQ job submitted. Job ID: 019a2a50-56de-75d8-8b2c-a40f04be9193
Using backend: <bound method BackendV1.name of <IonQQPUBackend('ionq_qpu.aria-1')>>
Submitted IonQ job ID: 019a2a50-56de-75d8-8b2c-a40f04be9193


The code below uses Measurement Device Independent (MDI) Quantum Key Distribution (QKD). In MDI-QKD rather than Bob performing the measurements, both Alice and Bob send a sequence of encoded qubits to an untrusted 3rd party Charlie. Charlie performs a Bell State Measurement, which checks if their qubits are correlated or anti-correlated and returns success or failure respectively. A Bell State Measurement does not ever require Charlie to know the qubit values meaning that even though he is not trusted it does not matter. 

In [5]:
# === Simplified Bell-state measurement ===
def bell_state_measurement(a_bit, a_basis, b_bit, b_basis) -> bool:
    """Return True if BSM successful; simplified probabilistic model"""
    if a_basis == b_basis:
        return random.choice([True, False])  # 50% success if bases match
    return False

# === Sifting for MDI-QKD ===
def mdi_sift(alice_bits, alice_bases, bob_bits, bob_bases):
    sifted_a, sifted_b, indices = [], [], []
    for i, (a_bit, a_basis, b_bit, b_basis) in enumerate(zip(alice_bits, alice_bases, bob_bits, bob_bases)):
        if bell_state_measurement(a_bit, a_basis, b_bit, b_basis):
            if a_basis == b_basis:
                sifted_a.append(a_bit)
                sifted_b.append(b_bit)
                indices.append(i)
    return sifted_a, sifted_b, indices

# === Main MDI-QKD simulation ===
def run_mdi_qkd(n: int = 8, noise_level: float = 0.0, noise_type: str = "depolarizing", verbose: bool = True):
    # Alice and Bob choose bits and bases
    alice_bits = random_bits(n)
    alice_bases = random_bases(n)
    bob_bits = random_bits(n)
    bob_bases = random_bases(n)

    # Prepare qubits
    alice_circuit = prepare_batch_circuit(alice_bits, alice_bases)
    bob_circuit   = prepare_batch_circuit(bob_bits, bob_bases)

    # Add noise if desired
    if noise_level > 0:
        alice_circuit = add_noise_to_circuit(alice_circuit, noise_level, noise_type)
        bob_circuit   = add_noise_to_circuit(bob_circuit, noise_level, noise_type)

    # Sifting based on simplified BSM
    sifted_a, sifted_b, indices = mdi_sift(alice_bits, alice_bases, bob_bits, bob_bases)
    err = error_rate(sifted_a, sifted_b)

    if verbose:
        print("=== MDI-QKD Simulation ===")
        print(f"Alice bits:  {alice_bits}")
        print(f"Alice bases: {alice_bases}")
        print(f"Bob bits:    {bob_bits}")
        print(f"Bob bases:   {bob_bases}")
        print(f"Sifted indices: {indices}")
        print(f"Sifted Alice:   {sifted_a}")
        print(f"Sifted Bob:     {sifted_b}")
        print(f"Error rate:     {err:.2f}")
        print(f"Key length:     {len(sifted_a)}")

    return {
        'alice_bits': alice_bits,
        'alice_bases': alice_bases,
        'bob_bits': bob_bits,
        'bob_bases': bob_bases,
        'sifted_alice': sifted_a,
        'sifted_bob': sifted_b,
        'sifted_indices': indices,
        'error_rate': err
    }

# === Test ===
result = run_mdi_qkd(n=8, noise_level=0.1, noise_type="depolarizing", verbose=True)

=== MDI-QKD Simulation ===
Alice bits:  [1, 1, 1, 0, 0, 1, 1, 1]
Alice bases: [0, 1, 0, 1, 1, 1, 1, 1]
Bob bits:    [0, 0, 1, 0, 0, 1, 0, 0]
Bob bases:   [0, 1, 1, 1, 0, 0, 1, 1]
Sifted indices: []
Sifted Alice:   []
Sifted Bob:     []
Error rate:     0.00
Key length:     0


In [6]:
# Poll IonQ job until completion and append results
from qiskit_ionq import IonQProvider
from dotenv import load_dotenv
import os, time


def poll_ionq_job(job_id: str, backend_name: str = "qpu.aria-1", interval: int = 15, timeout: int = 3600):
    """Poll an IonQ job until completion and print results.
    - job_id: IonQ job identifier (string)
    - backend_name: IonQ backend, default "qpu.aria-1"
    - interval: seconds between polls
    - timeout: max seconds to wait
    Returns a dict with counts and the most likely bitstring (q_0..q_n-1 order).
    """
    load_dotenv()
    key = os.getenv("IONQ_API_KEY")
    if not key:
        raise RuntimeError("IONQ_API_KEY not set. Put it in your .env or environment.")

    provider = IonQProvider(key)
    backend = provider.get_backend(backend_name)
    job = backend.retrieve_job(job_id)

    print(f"Polling IonQ job {job_id} on {backend_name}...")
    start = time.time()
    last_status = None
    while True:
        status = job.status()
        if status != last_status:
            print("Status:", status)
            last_status = status
        # Completed
        if str(status).upper() in ("DONE", "COMPLETED"):
            result = job.result()
            counts = result.get_counts()
            print("Counts:", counts)
            # Most frequent outcome as bitstring (Qiskit returns MSB->LSB; reverse for q_0..q_n-1)
            outcome = max(counts, key=counts.get)
            bits = [int(b) for b in outcome[::-1]]
            print("Most likely bitstring (q_0..q_n-1):", bits)
            return {"counts": counts, "bitstring": bits}
        # Failed/cancelled
        if str(status).upper() in ("ERROR", "CANCELLED"):
            raise RuntimeError(f"Job ended with status {status}")
        # Timeout
        if time.time() - start > timeout:
            raise TimeoutError("Timed out waiting for job to complete")
        time.sleep(interval)

# Usage:
# data = poll_ionq_job("019a2a2c-0c7f-74cf-ad33-d59be64a3ccd")
# data["counts"], data["bitstring"]

In [7]:
# End-to-end driver: submit -> poll (QPU) -> fallback to simulator if needed -> sift -> cascade -> privacy amp

def most_likely_bits_from_counts(counts: dict) -> list:
    outcome = max(counts, key=counts.get)
    return [int(b) for b in outcome[::-1]]  # q_0..q_{n-1}

print("\n=== Submitting BB84 to IonQ QPU and running full post-processing ===")
ctx = run_bb84(
    n=8,
    with_eve=False,
    verbose=True,
    print_circuits=True,
    noise_level=0.0,
    noise_type="depolarizing",
    cascade=True,
    privacy_amp=True,
)

counts = None
bob_bits = None
job_id = None

if flag:
    job_id = ctx["job_id"]
    print(f"\nPolling job until completion: {job_id}")
    try:
        data = poll_ionq_job(job_id, backend_name="qpu.aria-1", interval=20, timeout=5400)
        counts = data["counts"]
        bob_bits = most_likely_bits_from_counts(counts)
    except Exception as e:
        print("\n[Fallback] QPU polling failed:", e)
        print("Running the same BB84 context on IonQ simulator to complete the process.")
        # Rebuild measurement circuit from saved context
        alice_bits = ctx["alice_bits"]
        alice_bases = ctx["alice_bases"]
        bob_bases = ctx["bob_bases"]
        forwarded_circuit = prepare_batch_circuit(alice_bits, alice_bases)
        bob_measure_circuit = measure_batch_circuit(forwarded_circuit, bob_bases)
        # Temporarily switch backend to simulator
        _old_backend = backend
        _old_flag = flag
        backend = provider.get_backend("simulator")
        flag = 0
        try:
            bob_bits = run_circuit_get_bits(bob_measure_circuit)
            # Synthesize a counts dict for display
            bitstr = ''.join(str(b) for b in bob_bits[::-1])
            counts = {bitstr: 1}
        finally:
            backend = _old_backend
            flag = _old_flag
else:
    print("Flag indicates simulator path; no polling required in this branch.")

# If we have bob_bits (from QPU or simulator), finish protocol
if bob_bits is not None:
    alice_bits = ctx["alice_bits"]
    alice_bases = ctx["alice_bases"]
    bob_bases = ctx["bob_bases"]

    sifted_a, sifted_b, indices = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    err_before = error_rate(sifted_a, sifted_b)

    # Cascade EC
    sifted_a_corr, sifted_b_corr, corrections = cascade_error_correction(sifted_a, sifted_b, num_passes=4)
    err_after = error_rate(sifted_a_corr, sifted_b_corr)

    # Privacy amplification
    final_key = privacy_amplification(sifted_a_corr)

    print("\n=== Summary ===")
    print(f"Backend: {backend.name}")
    if job_id:
        print(f"Job ID: {job_id}")
    print(f"Counts: {counts}")
    print(f"Bob bits (ML): {bob_bits}")
    print(f"Sifted indices: {indices}")
    print(f"Sifted Alice: {sifted_a}")
    print(f"Sifted Bob:   {sifted_b}")
    print(f"Error rate before Cascade: {err_before}")
    print(f"Corrections applied: {corrections}")
    print(f"Error rate after Cascade:  {err_after}")
    print(f"Final key (privacy amp):   {final_key}")
    print(f"Final key length:          {len(final_key)}")
else:
    print("No measurement results available.")


=== Submitting BB84 to IonQ QPU and running full post-processing ===

COMPLETE BB84 CIRCUIT DIAGRAM:
                ░ ┌───┐            ┌─┐         
q_0: ───────────░─┤ H ├────────────┤M├─────────
     ┌───┐┌───┐ ░ └───┘┌─┐         └╥┘         
q_1: ┤ X ├┤ H ├─░──────┤M├──────────╫──────────
     ├───┤├───┤ ░ ┌───┐└╥┘          ║ ┌─┐      
q_2: ┤ X ├┤ H ├─░─┤ H ├─╫───────────╫─┤M├──────
     ├───┤└───┘ ░ ├───┤ ║           ║ └╥┘┌─┐   
q_3: ┤ H ├──────░─┤ H ├─╫───────────╫──╫─┤M├───
     └───┘      ░ └───┘ ║ ┌─┐       ║  ║ └╥┘   
q_4: ───────────░───────╫─┤M├───────╫──╫──╫────
                ░       ║ └╥┘┌─┐    ║  ║  ║    
q_5: ───────────░───────╫──╫─┤M├────╫──╫──╫────
     ┌───┐      ░       ║  ║ └╥┘┌─┐ ║  ║  ║    
q_6: ┤ X ├──────░───────╫──╫──╫─┤M├─╫──╫──╫────
     ├───┤      ░ ┌───┐ ║  ║  ║ └╥┘ ║  ║  ║ ┌─┐
q_7: ┤ X ├──────░─┤ H ├─╫──╫──╫──╫──╫──╫──╫─┤M├
     └───┘      ░ └───┘ ║  ║  ║  ║  ║  ║  ║ └╥┘
c: 8/═══════════════════╩══╩══╩══╩══╩══╩══╩══╩═
                        1  4  5  6

/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for mcx can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for cnot can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for toffoli can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153:

IonQ job submitted. Job ID: 019a2a50-57f0-742b-b6e7-653f3ebb3f1e
Using backend: <bound method BackendV1.name of <IonQQPUBackend('ionq_qpu.aria-1')>>
Submitted IonQ job ID: 019a2a50-57f0-742b-b6e7-653f3ebb3f1e

Polling job until completion: 019a2a50-57f0-742b-b6e7-653f3ebb3f1e


Polling IonQ job 019a2a50-57f0-742b-b6e7-653f3ebb3f1e on qpu.aria-1...
Status: JobStatus.INITIALIZING



[Fallback] QPU polling failed: IonQJobFailureError('Unable to retrieve result for job 019a2a50-57f0-742b-b6e7-653f3ebb3f1e. Failure from IonQ API "QuotaExhaustedError: "')
Running the same BB84 context on IonQ simulator to complete the process.


/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for mcx can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for cnot can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for toffoli can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153:


=== Summary ===
Backend: <bound method BackendV1.name of <IonQQPUBackend('ionq_qpu.aria-1')>>
Job ID: 019a2a50-57f0-742b-b6e7-653f3ebb3f1e
Counts: {'11000101': 1}
Bob bits (ML): [1, 0, 1, 0, 0, 0, 1, 1]
Sifted indices: [2, 3, 4, 5, 6]
Sifted Alice: [1, 0, 0, 0, 1]
Sifted Bob:   [1, 0, 0, 0, 1]
Error rate before Cascade: 0.0
Corrections applied: 0
Error rate after Cascade:  0.0
Final key (privacy amp):   [1, 1]
Final key length:          2


In [8]:
import numpy as np
from math import log2
from sklearn.svm import SVR, SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
try:
    import tensorflow as tf
    from tensorflow import keras
    TF_AVAILABLE = True
except Exception:
    TF_AVAILABLE = False

In [9]:
# Synthetic dataset generation using IonQ simulator for quick iterations
from typing import Dict

def approx_secret_key_rate(sifted_len: int, qber: float) -> float:
    """Very rough key rate proxy: sifted_len * max(0, 1 - 2*H2(qber))."""
    def H2(p):
        if p <= 0 or p >= 1:
            return 0.0
        return -p*log2(p) - (1-p)*log2(1-p)
    return max(0.0, sifted_len * (1 - 2*H2(qber)))


def simulate_bb84_once(n: int, noise_level: float, with_eve: bool) -> Dict:
    global backend, flag
    _old_backend = backend
    _old_flag = flag
    try:
        backend = provider.get_backend("simulator")
        flag = 0
        res = run_bb84(n=n, with_eve=with_eve, verbose=False, print_circuits=False, noise_level=noise_level, cascade=True, privacy_amp=False)
        sifted_a = res['sifted_alice']
        sifted_b = res['sifted_bob']
        qber = error_rate(sifted_a, sifted_b)
        rate = approx_secret_key_rate(len(sifted_a), qber)
        return {
            'n': n,
            'noise': noise_level,
            'with_eve': int(with_eve),
            'qber': qber,
            'sifted_len': len(sifted_a),
            'key_rate': rate,
            'protocol': 'BB84'
        }
    finally:
        backend = _old_backend
        flag = _old_flag


def simulate_mdi_once(n: int, noise_level: float) -> Dict:
    # Use existing MDI simulation (no hardware dependency)
    r = run_mdi_qkd(n=n, noise_level=noise_level, verbose=False)
    qber = r['error_rate']
    rate = approx_secret_key_rate(len(r['sifted_alice']), qber)
    return {
        'n': n,
        'noise': noise_level,
        'with_eve': 0,
        'qber': qber,
        'sifted_len': len(r['sifted_alice']),
        'key_rate': rate,
        'protocol': 'MDI'
    }

# Build dataset
import random as _rnd

def build_dataset(samples: int = 60, n_choices=(4,6,8), noise_range=(0.0, 0.2)):
    data = []
    for _ in range(samples):
        n = _rnd.choice(n_choices)
        noise = _rnd.uniform(*noise_range)
        # BB84 with and without Eve
        data.append(simulate_bb84_once(n, noise, with_eve=False))
        data.append(simulate_bb84_once(n, noise, with_eve=True))
        # MDI
        data.append(simulate_mdi_once(n, noise))
    return data

print("Generating synthetic dataset (this may take ~10-30s)...")
dataset = build_dataset(samples=40)
print(f"Dataset size: {len(dataset)} rows")

Generating synthetic dataset (this may take ~10-30s)...

ADDING SIMULATED NOISE TO CHANNEL
Noise level: 0.01, Type: depolarizing


/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for mcx can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for cnot can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for toffoli can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153:


EVE INTERCEPT-RESEND ATTACK:
Eve's bases: [0, 0, 0, 0, 1, 0]


/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for mcx can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for cnot can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for toffoli can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153:


ADDING SIMULATED NOISE TO CHANNEL
Noise level: 0.01, Type: depolarizing


...


EVE INTERCEPT-RESEND ATTACK:
Eve's bases: [0, 1, 1, 0]


/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for mcx can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for cnot can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for toffoli can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153:


ADDING SIMULATED NOISE TO CHANNEL
Noise level: 0.14, Type: depolarizing


/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for mcx can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for cnot can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153: RuntimeWarning: No gate definition for toffoli can be found and is being excluded from the generated target. You can use `custom_name_mapping` to provide a definition for this operation.
  warnings.warn(
/Users/jasonli/Library/Python/3.9/lib/python/site-packages/qiskit/providers/backend_compat.py:153:

Dataset size: 120 rows


In [10]:
# Train SVR: predict key_rate from [n, noise, with_eve]
import numpy as _np

# Prepare feature matrix and target
feat_cols = ["n", "noise", "with_eve"]
X = _np.array([[row[c] for c in feat_cols] for row in dataset], dtype=float)
y = _np.array([row['key_rate'] for row in dataset], dtype=float)

svr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=10.0, gamma="scale", epsilon=0.1))
])
svr_model.fit(X, y)

print("SVR trained. Example prediction (n=8, noise=0.05, with_eve=0):",
      svr_model.predict([[8, 0.05, 0]]))

SVR trained. Example prediction (n=8, noise=0.05, with_eve=0): [1.40326306]


In [11]:
# Protocol selector: choose best protocol (BB84 vs MDI) for given [n, noise, with_eve]
from collections import defaultdict

# Build paired comparisons
pairs = defaultdict(dict)
for row in dataset:
    key = (row['n'], round(row['noise'], 3), int(row['with_eve']))
    pairs[key][row['protocol']] = row['key_rate']

X_proto = []
y_proto = []  # 0 -> BB84, 1 -> MDI
for (n, noise, we), ks in pairs.items():
    bb84 = ks.get('BB84', -1)
    mdi = ks.get('MDI', -1)
    if bb84 < 0 and mdi < 0:
        continue
    best = 0 if bb84 >= mdi else 1
    X_proto.append([n, noise, we])
    y_proto.append(best)

proto_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=5.0, gamma="scale", probability=True))
])
proto_clf.fit(X_proto, y_proto)

print("Protocol selector trained. Example (n=8, noise=0.05, with_eve=1):",
      ['BB84','MDI'][int(proto_clf.predict([[8,0.05,1]])[0])])

Protocol selector trained. Example (n=8, noise=0.05, with_eve=1): BB84


In [12]:
# MLPRegressor: real-time key rate prediction from richer features
features_full = ["n", "noise", "with_eve", "qber", "sifted_len"]
X_full = _np.array([[row.get(c, 0.0) for c in features_full] for row in dataset], dtype=float)
y_full = _np.array([row['key_rate'] for row in dataset], dtype=float)

mlp_model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu', max_iter=1000, random_state=42))
])
mlp_model.fit(X_full, y_full)

print("MLP trained. Example prediction (n=8, noise=0.05, with_eve=0, qber=0.02, sifted_len=4):",
      mlp_model.predict([[8, 0.05, 0, 0.02, 4]]) )

MLP trained. Example prediction (n=8, noise=0.05, with_eve=0, qber=0.02, sifted_len=4): [3.37536316]


In [13]:
# ML-assisted recommendation demo

def recommend_settings(n: int, noise: float, with_eve: int):
    # Predict key rate via SVR
    pred_rate = float(svr_model.predict([[n, noise, with_eve]])[0])
    # Choose protocol
    proto_idx = int(proto_clf.predict([[n, noise, with_eve]])[0])
    protocol = ['BB84','MDI'][proto_idx]
    # Real-time key rate prediction with assumed qber and sifted length
    # Use simple priors based on noise and n
    est_qber = min(0.5, max(0.0, noise * 2.5 + 0.02*with_eve))
    est_sifted = max(1, int(n * (0.5 if protocol=='BB84' else 0.3)))
    pred_rt = float(mlp_model.predict([[n, noise, with_eve, est_qber, est_sifted]])[0])
    return {
        'recommended_protocol': protocol,
        'pred_key_rate_svr': pred_rate,
        'pred_key_rate_realtime': pred_rt,
        'assumed_qber': est_qber,
        'assumed_sifted_len': est_sifted,
    }

print("Recommendation example (n=8, noise=0.05, with_eve=1):")
print(recommend_settings(8, 0.05, 1))

Recommendation example (n=8, noise=0.05, with_eve=1):
{'recommended_protocol': 'BB84', 'pred_key_rate_svr': 0.15267524194317472, 'pred_key_rate_realtime': 1.4349898283154803, 'assumed_qber': 0.145, 'assumed_sifted_len': 4}
